In [1]:
import json
import numpy as np

In [2]:
class LoadData:
    def __init__(self, file_path):
        with open(file_path, 'r') as f:
            self.data = json.load(f)
    
    def create_data(self):
        contexts = []
        questions = []
        answers = []
        for group in self.data['data']:
            for passage in group['paragraphs']:
                context = passage['context']
                for qa in passage['qas']:
                    question = qa['question']
                    for answer in qa['answers']:
                        answers.append(answer)
                        questions.append(question)
                        contexts.append(context)

        return answers, questions, contexts

In [1]:
import json
with open('data/train-v2.0.json', 'r') as f:
    data = json.load(f) 

In [9]:
data['data'][0]['paragraphs'][0]['qas'][0]

{'question': 'When did Beyonce start becoming popular?',
 'id': '56be85543aeaaa14008c9063',
 'answers': [{'text': 'in the late 1990s', 'answer_start': 269}],
 'is_impossible': False}

In [3]:
class PreprocessData:
    def __init__(self, tokenizer, answers, questions, contexts):
        self.contexts = contexts
        self.questions = questions
        self.answers = answers
        self.tokenizer = tokenizer

    # def reg_tokenize(self, text):
    #     WORD = re.compile(r'\w+')
    #     words = WORD.findall(text)
    #     return words

    def create_encodings(self):
        encodings = self.tokenizer(self.contexts, self.questions, truncation=True, padding=True)
        self.__add_answer_end_pos()
        encodings = self.__convert_to_token_start_end_pos(encodings)
        # encodings = self.__create_labels(encodings)
        return encodings
        
    def __add_answer_end_pos(self):
        for answer, text in zip(self.answers, self.contexts):
            real_answer = answer['text']
            start_idx = answer['answer_start']
            # Get the real end index
            end_idx = start_idx + len(real_answer)

            # Deal with the problem of 1 or 2 more characters 
            if text[start_idx:end_idx] == real_answer:
                answer['answer_end'] = end_idx
            # When the real answer is more by one character
            elif text[start_idx-1:end_idx-1] == real_answer:
                answer['answer_start'] = start_idx - 1
                answer['answer_end'] = end_idx - 1  
            # When the real answer is more by two characters  
            elif text[start_idx-2:end_idx-2] == real_answer:
                answer['answer_start'] = start_idx - 2
                answer['answer_end'] = end_idx - 2 

    def __convert_to_token_start_end_pos(self, encodings):
        start_token_positions = []
        end_token_positions = []
        for index, answer in enumerate(self.answers):
            start = encodings.char_to_token(index, answer['answer_start'])
            end = encodings.char_to_token(index, answer['answer_end'])

            # if start = None, the answers have been truncated
            if start == None:
                start = self.tokenizer.model_max_length

            # if end == None, the 'char_to_token' function points to the space after the correct token, so add - 1
            if end == None:
                end = encodings.char_to_token(index, answer['answer_end'] - 1)
                # if end is still None, the answers have been truncated
                if end == None:
                    end = self.tokenizer.model_max_length

            start_token_positions.append(start)
            end_token_positions.append(end)

        encodings['start_positions'] = start_token_positions
        encodings['end_positions'] = end_token_positions

        return encodings

    def __create_labels(self, encodings):
        encodings['answer_length'] = np.array(encodings['end_positions'])\
         - np.array(encodings['start_positions']) + 1
        labels = np.zeros((len(self.answers), self.tokenizer.model_max_length, 
            2)) # num_example * seq_length * 2

        for example_idx, start in enumerate(encodings['start_positions']):
            if start < self.tokenizer.model_max_length: # if the answer is not truncated
                labels[example_idx, start, 0] = 1
                labels[example_idx, start, 1] = encodings['answer_length'][example_idx]

        encodings['labels'] = labels

        return encodings

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

d:\ComputerScience\BachKhoa\ProjectII\YOLOQA\venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from pathlib import Path
# path = Path('data/train-v2.0.json')
# path = Path('data/debug.json')
path = Path('data/dev-v2.0.json')
load_data = LoadData(path)
answers, questions, contexts = load_data.create_data()
preprocess = PreprocessData(tokenizer, answers, questions, contexts)

In [6]:
len(preprocess.answers)

20302

In [7]:
preprocess.answers[0]

{'text': 'France', 'answer_start': 159}

In [8]:
# preprocess.create_encodings()
# preprocess.add_answer_end_pos()
# preprocess.convert_to_token_start_end_pos()
# labels = preprocess.create_labels()
encodings = preprocess.create_encodings()

In [9]:
encodings.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'])

In [10]:
len(encodings['input_ids'])

20302

In [23]:
# labels[0, 67, :]

In [11]:
preprocess.tokenizer.convert_ids_to_tokens(encodings['input_ids'][0])

['[CLS]',
 'the',
 'norman',
 '##s',
 '(',
 'norman',
 ':',
 'no',
 '##ur',
 '##man',
 '##ds',
 ';',
 'french',
 ':',
 'norman',
 '##ds',
 ';',
 'latin',
 ':',
 'norman',
 '##ni',
 ')',
 'were',
 'the',
 'people',
 'who',
 'in',
 'the',
 '10th',
 'and',
 '11th',
 'centuries',
 'gave',
 'their',
 'name',
 'to',
 'normandy',
 ',',
 'a',
 'region',
 'in',
 'france',
 '.',
 'they',
 'were',
 'descended',
 'from',
 'norse',
 '(',
 '"',
 'norman',
 '"',
 'comes',
 'from',
 '"',
 'norse',
 '##man',
 '"',
 ')',
 'raiders',
 'and',
 'pirates',
 'from',
 'denmark',
 ',',
 'iceland',
 'and',
 'norway',
 'who',
 ',',
 'under',
 'their',
 'leader',
 'roll',
 '##o',
 ',',
 'agreed',
 'to',
 'swear',
 'fe',
 '##al',
 '##ty',
 'to',
 'king',
 'charles',
 'iii',
 'of',
 'west',
 'fran',
 '##cia',
 '.',
 'through',
 'generations',
 'of',
 'assimilation',
 'and',
 'mixing',
 'with',
 'the',
 'native',
 'frankish',
 'and',
 'roman',
 '-',
 'gaul',
 '##ish',
 'populations',
 ',',
 'their',
 'descendants',


In [12]:
# import pickle
# with open('data/train_preprocessed2.pkl', 'wb') as f:
#     pickle.dump(encodings, f)

import pickle
with open('data/dev_preprocessed.pkl', 'wb') as f:
    pickle.dump(encodings, f)

In [13]:
encodings.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'])

In [14]:
len(encodings['input_ids'])

20302